# Analisis exploratorio de datos - Heart Disease (UCI)

Practica desarrollada segun la guia del Word: carga de datos, exploracion inicial, resumen estadistico, visualizacion, frecuencias categoricas, correlacion, manejo de faltantes, outliers, segmentacion, hipotesis y presentacion de resultados.

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target_raw"
]

## 1. Carga de datos

In [ ]:
df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)
df = df.replace("?", np.nan)
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["target"] = (df["target_raw"] > 0).astype(int)

print("Filas, columnas:", df.shape)
df.head()

## 2. Exploracion inicial y calidad de datos

In [ ]:
print("Tipos de datos:")
print(df.dtypes)

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

duplicates = int(df.duplicated().sum())

with open(os.path.join(OUTPUT_DIR, "01_exploracion_inicial.txt"), "w", encoding="utf-8") as f:
    f.write("Exploracion inicial - Heart Disease UCI\n")
    f.write(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Fuente: {DATA_URL}\n\n")
    f.write(f"Observaciones: {df.shape[0]}\n")
    f.write(f"Variables: {df.shape[1]}\n")
    f.write("Variable objetivo Y: target (0 sin enfermedad, 1 con enfermedad)\n")

missing.to_csv(os.path.join(OUTPUT_DIR, "02_valores_faltantes.csv"), encoding="utf-8")
with open(os.path.join(OUTPUT_DIR, "03_duplicados.txt"), "w", encoding="utf-8") as f:
    f.write(f"Filas duplicadas: {duplicates}\n")

missing.head(10)

## 3. Resumen estadistico

In [ ]:
stats = df.describe(include=["number"]).T
stats.to_csv(os.path.join(OUTPUT_DIR, "04_resumen_estadistico.csv"), encoding="utf-8")
stats

## 4. Analisis de variables categoricas (frecuencias)

In [ ]:
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal", "target"]
rows = []
for col in cat_cols:
    vc = df[col].value_counts(dropna=False).sort_index()
    for k, v in vc.items():
        rows.append({"variable": col, "categoria": k, "frecuencia": int(v)})

freq_df = pd.DataFrame(rows)
freq_df.to_csv(os.path.join(OUTPUT_DIR, "05_frecuencias_categoricas.csv"), index=False)
freq_df.head(20)

## 5. Visualizacion de datos

In [ ]:
# Histograma
plt.figure(figsize=(8, 4))
sns.histplot(df["age"].dropna(), kde=True, bins=20, color="#1f77b4")
plt.title("Distribucion de edad")
plt.xlabel("Edad")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_hist_age.png"), dpi=150)
plt.show()

# Scatter
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="thalach", y="oldpeak", hue="target", palette="Set2", alpha=0.85)
plt.title("Relacion entre thalach y oldpeak por target")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "02_scatter_thalach_oldpeak.png"), dpi=150)
plt.show()

# Barras categoricas
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="cp", hue="target", palette="Set1")
plt.title("Frecuencia de cp por target")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "03_bar_cp_target.png"), dpi=150)
plt.show()

# Boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="target", y="chol", color="#9ecae1")
plt.title("Outliers de colesterol por target")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "04_boxplot_chol_target.png"), dpi=150)
plt.show()

## 6. Manejo de datos faltantes

In [ ]:
df_treated = df.copy()
missing_before = int(df_treated.isna().sum().sum())

if df_treated["ca"].isna().any():
    df_treated["ca"] = df_treated["ca"].fillna(df_treated["ca"].median())
if df_treated["thal"].isna().any():
    df_treated["thal"] = df_treated["thal"].fillna(df_treated["thal"].mode().iloc[0])

missing_after = int(df_treated.isna().sum().sum())
df_treated.to_csv(os.path.join(OUTPUT_DIR, "07_dataset_tratado.csv"), index=False)

print("Faltantes antes:", missing_before)
print("Faltantes despues:", missing_after)

## 7. Analisis de correlacion

In [ ]:
corr = df_treated.corr(numeric_only=True)
corr.to_csv(os.path.join(OUTPUT_DIR, "08_correlaciones.csv"), encoding="utf-8")

excluded = ["target", "target_raw"]
present = [c for c in excluded if c in corr.columns]
target_corr = corr["target"].drop(labels=present).sort_values(key=lambda s: s.abs(), ascending=False)
top3 = target_corr.head(3).reset_index()
top3.columns = ["variable", "corr_with_target"]
top3.to_csv(os.path.join(OUTPUT_DIR, "09_top3_correlacion_Y.csv"), index=False)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matriz de correlacion")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "05_heatmap_correlacion.png"), dpi=150)
plt.show()

top3

## 8. Analisis de outliers (IQR)

In [ ]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
out_rows = []

for col in numeric_cols:
    s = df_treated[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = int(((s < lower) | (s > upper)).sum())
    pct = round((n_out / len(s)) * 100, 2) if len(s) else 0.0
    out_rows.append({
        "variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower": lower, "upper": upper,
        "n_outliers": n_out, "outlier_pct": pct
    })

out_df = pd.DataFrame(out_rows).sort_values("outlier_pct", ascending=False)
out_df.to_csv(os.path.join(OUTPUT_DIR, "06_outliers_iqr.csv"), index=False)
out_df

## 9. Segmentacion

In [ ]:
seg = df_treated.copy()
seg["age_group"] = pd.cut(seg["age"], bins=[0, 39, 49, 59, 69, 120], labels=["<=39", "40-49", "50-59", "60-69", ">=70"])

by_age = seg.groupby("age_group", observed=False)["target"].mean().reset_index()
by_age.columns = ["age_group", "target_rate"]
by_sex = seg.groupby("sex", observed=False)["target"].mean().reset_index()
by_sex.columns = ["sex", "target_rate"]

by_age.to_csv(os.path.join(OUTPUT_DIR, "10_segmentacion_age_group.csv"), index=False)
by_sex.to_csv(os.path.join(OUTPUT_DIR, "11_segmentacion_sex.csv"), index=False)

print("Segmentacion por edad:")
display(by_age)
print("Segmentacion por sexo:")
display(by_sex)

## 10. Hipotesis iniciales

1. Pacientes con menor thalach y mayor oldpeak tendran mayor probabilidad de target=1.
2. Ciertas categorias de cp y exang se asocian con mayor riesgo cardiaco.
3. Segmentos de mayor edad presentaran mayor prevalencia de enfermedad cardiaca.

## 11. Presentacion de resultados

Para la entrega, usar los archivos exportados en output/ y las figuras en output/plots/.

Este notebook queda listo para subir al AVAC junto con la presentacion PDF o PPTX.